# 07 — TabNet Model

**TabNet** is an attentive tabular neural net (Arik & Pfister). We train it on the **winning GA/PSO feature subset** (primary) and once on **all features** for a short before/after comparison.

In [1]:
from pathlib import Path
import random

import numpy as np

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [cwd, *cwd.parents] if (p / "environment.yml").exists()),
    cwd,
)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

for d in (DATA_INTERIM, DATA_PROCESSED, FIGURES_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Random seed  : {RANDOM_SEED}")

Project root : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction
Random seed  : 42


In [2]:
import json
import time
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
)

X_train = pd.read_csv(DATA_PROCESSED / "X_train.csv")
X_test = pd.read_csv(DATA_PROCESSED / "X_test.csv")
y_train = pd.read_csv(DATA_PROCESSED / "y_train.csv").squeeze().values
y_test = pd.read_csv(DATA_PROCESSED / "y_test.csv").squeeze().values
winner = json.loads((DATA_PROCESSED / "nia_feature_selection_winner.json").read_text(encoding="utf-8"))
win_feats = winner["winning_features"]
print("Winner:", winner["winner"], "n_features", len(win_feats))

# Class weights for imbalance
n_pos = (y_train == 1).sum()
n_neg = (y_train == 0).sum()
w_pos = n_neg / max(n_pos, 1)
print("pos weight ~", w_pos)

Winner: GA n_features 17
pos weight ~ 4.941952506596306


In [3]:
def eval_preds(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, proba)),
        "pr_auc": float(average_precision_score(y_true, proba)),
    }

def train_tabnet(X_tr, y_tr, X_te, y_te, name):
    clf = TabNetClassifier(
        n_d=16, n_a=16, n_steps=3, gamma=1.5,
        lambda_sparse=1e-4, optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-2),
        scheduler_params={"step_size": 10, "gamma": 0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type="entmax",
        verbose=0,
        seed=RANDOM_SEED,
    )
    t0 = time.perf_counter()
    clf.fit(
        X_tr, y_tr,
        eval_set=[(X_te, y_te)],
        eval_metric=["auc", "logloss"],
        max_epochs=100,
        patience=15,
        batch_size=256,
        virtual_batch_size=128,
        weights=1,  # balance
    )
    elapsed = time.perf_counter() - t0
    proba = clf.predict_proba(X_te)[:, 1]
    metrics = eval_preds(y_te, proba)
    metrics["model"] = name
    metrics["n_features"] = int(X_tr.shape[1])
    metrics["train_time_sec"] = float(elapsed)
    return clf, metrics, proba

# Winning subset
Xtr_w = X_train[win_feats].values.astype(np.float32)
Xte_w = X_test[win_feats].values.astype(np.float32)
clf_w, metrics_w, proba_w = train_tabnet(Xtr_w, y_train, Xte_w, y_test, "TabNet_winning_subset")
print(metrics_w)

# Full features
Xtr_f = X_train.values.astype(np.float32)
Xte_f = X_test.values.astype(np.float32)
clf_f, metrics_f, proba_f = train_tabnet(Xtr_f, y_train, Xte_f, y_test, "TabNet_all_features")
print(metrics_f)


Early stopping occurred at epoch 65 with best_epoch = 50 and best_val_0_logloss = 0.02471


C:\Users\ishan\AppData\Roaming\Python\Python313\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


{'accuracy': 0.9911190053285968, 'precision': 0.95, 'recall': 1.0, 'f1': 0.9743589743589743, 'roc_auc': 0.9999662618083671, 'pr_auc': 0.9998359330532275, 'model': 'TabNet_winning_subset', 'n_features': 17, 'train_time_sec': 28.33866019999914}



Early stopping occurred at epoch 89 with best_epoch = 74 and best_val_0_logloss = 0.02916


C:\Users\ishan\AppData\Roaming\Python\Python313\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


{'accuracy': 0.9902309058614565, 'precision': 0.9543147208121827, 'recall': 0.9894736842105263, 'f1': 0.9715762273901809, 'roc_auc': 0.9997750787224471, 'pr_auc': 0.998944517887541, 'model': 'TabNet_all_features', 'n_features': 33, 'train_time_sec': 38.50664710000274}


In [4]:
cmp = pd.DataFrame([metrics_w, metrics_f])
cmp.to_json(REPORTS_DIR / "07_tabnet_metrics.json", orient="records", indent=2)
print(cmp)

# Save primary model (winning subset)
save_dir = MODELS_DIR / "tabnet_winning"
save_dir.mkdir(exist_ok=True)
clf_w.save_model(str(save_dir / "model"))
joblib.dump({"features": win_feats, "metrics": metrics_w, "winner_algo": winner["winner"]}, MODELS_DIR / "tabnet_meta.joblib")

# Feature importance from TabNet (winning)
imp = pd.DataFrame({"feature": win_feats, "importance": clf_w.feature_importances_}).sort_values("importance", ascending=False)
imp.to_csv(DATA_PROCESSED / "07_tabnet_feature_importance.csv", index=False)

fig, ax = plt.subplots(figsize=(8, max(3, len(win_feats) * 0.25)))
ax.barh(imp["feature"], imp["importance"], color="#264653")
ax.invert_yaxis()
ax.set_title("TabNet feature importance (winning subset)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "07_tabnet_importance.png", dpi=150)
plt.show()

cm = confusion_matrix(y_test, (proba_w >= 0.5).astype(int))
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, str(v), ha="center", va="center")
ax.set_title("TabNet (winning) confusion matrix")
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "07_tabnet_confusion.png", dpi=150)
plt.show()

np.save(DATA_PROCESSED / "tabnet_test_proba_winning.npy", proba_w)

   accuracy  precision    recall        f1   roc_auc    pr_auc  \
0  0.991119   0.950000  1.000000  0.974359  0.999966  0.999836   
1  0.990231   0.954315  0.989474  0.971576  0.999775  0.998945   

                   model  n_features  train_time_sec  
0  TabNet_winning_subset          17       28.338660  
1    TabNet_all_features          33       38.506647  
Successfully saved model at C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction\models\tabnet_winning\model.zip


C:\Users\ishan\AppData\Local\Temp\ipykernel_24960\2569557489.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\ishan\AppData\Local\Temp\ipykernel_24960\2569557489.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# Visuals: TabNet subset vs full (from saved metrics + test probabilities)
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

cmp = pd.read_json(REPORTS_DIR / "07_tabnet_metrics.json")
metrics_w = cmp.iloc[0].to_dict()
metrics_f = cmp.iloc[1].to_dict()
y_test = pd.read_csv(DATA_PROCESSED / "y_test.csv").squeeze().values
proba_w = np.load(DATA_PROCESSED / "tabnet_test_proba_winning.npy")

keys = ["f1", "recall", "pr_auc"]
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(keys))
w = 0.35
ax.bar(x - w / 2, [metrics_w[k] for k in keys], width=w, label="Winning subset", color="#2a9d8f")
ax.bar(x + w / 2, [metrics_f[k] for k in keys], width=w, label="All features", color="#8d99ae")
ax.set_xticks(x)
ax.set_xticklabels(["F1", "Recall", "PR-AUC"])
ax.set_ylim(0, 1.05)
ax.set_title("TabNet: GA/PSO subset vs all features")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "07_tabnet_metric_compare.png", dpi=150)
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
RocCurveDisplay.from_predictions(y_test, proba_w, name="Winning subset", ax=ax[0])
PrecisionRecallDisplay.from_predictions(y_test, proba_w, name="Winning subset", ax=ax[1])
ax[0].set_title("TabNet ROC (winning subset)")
ax[1].set_title("TabNet Precision-Recall (winning subset)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "07_tabnet_roc_pr.png", dpi=150)
plt.show()
print(cmp[["model", "f1", "recall", "pr_auc", "n_features"]])


**Insight (simple):** If the green bars (winning subset) match or beat the grey bars (all features), GA/PSO helped TabNet by removing noise while keeping signal.

**Interpretation:** Compare F1 / PR-AUC / Recall between winning-subset TabNet and all-features TabNet. If the subset wins or ties with fewer inputs, GA/PSO delivered a real reduction benefit for the neural predictor.

**Next:** Notebook `08` — SHAP explainability.